In [1]:
import os
import sys

os.environ["JAVA_HOME"] = (
    r"C:\Program Files\zulu17.70.201-ca-jdk17.0.21.1-win_x64"
    r"\zulu17.70.201-ca-jdk17.0.21.1-win_x64"
)

os.environ["HADOOP_HOME"] = r"C:\hadoop-3.5.0\hadoop-3.5.0"

os.environ["PATH"] = (
    os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("NYC-Taxi-Kafka-Delta")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

builder = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    ]
)

spark = builder.getOrCreate()

In [2]:
print("Spark version:", spark.version)
print("Delta version:", spark.conf.get("spark.sql.extensions"))
print("Kafka version:", spark.conf.get("spark.jars.packages"))
print("Java version:", os.environ["JAVA_HOME"])
print("Hadoop version:", os.environ["HADOOP_HOME"])
print("Python version:", sys.version)
print("PySpark version:", spark.sparkContext.version)
print("Spark session:", spark.sparkContext.appName)

Spark version: 4.2.0
Delta version: io.delta.sql.DeltaSparkSessionExtension
Kafka version: io.delta:delta-spark_4.2_2.13:4.4.1,org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0
Java version: C:\Program Files\zulu17.70.201-ca-jdk17.0.21.1-win_x64\zulu17.70.201-ca-jdk17.0.21.1-win_x64
Hadoop version: C:\hadoop-3.5.0\hadoop-3.5.0
Python version: 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
PySpark version: 4.2.0
Spark session: NYC-Taxi-Kafka-Delta


Kafka command cd G:\kafka_2.13-4.4.0
$env:JAVA_HOME = "C:\Program Files\zulu17.70.201-ca-jdk17.0.21.1-win_x64\zulu17.70.201-ca-jdk17.0.21.1-win_x64"

$env:PATH = "$env:JAVA_HOME\bin;$env:PATH"

in powersehll one cd G:\kafka_2.13-4.4.0
run kafka 
.\bin\windows\kafka-server-start.bat .\config\server.properties 

in powershell 2 start topic

verify kafka is running 

Get-NetTCPConnection -LocalPort 9092,9094 -State Listen |
Select-Object LocalAddress, LocalPort, OwningProcess

.\bin\windows\kafka-topics.bat --create --topic taxi-trips-full --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1

.\bin\windows\kafka-topics.bat --list --bootstrap-server localhost:9092

.\bin\windows\kafka-topics.bat --delete --topic taxi-trips --bootstrap-server localhost:9092


to stop kafka ctrl+c in powershell 1

PS G:\kafka_2.13-4.4.0> Get-ChildItem $path |
>> Where-Object { $_.Name -like "taxi-trips*" } |
>> Select-Object Name, FullName

Name              FullName
----              --------
taxi-trips-full-0 G:\kafka_2.13-4.4.0\data\kraft-logs\taxi-trips-full-0
taxi-trips-full-1 G:\kafka_2.13-4.4.0\data\kraft-logs\taxi-trips-full-1
taxi-trips-full-2 G:\kafka_2.13-4.4.0\data\kraft-logs\taxi-trips-full-2


PS G:\kafka_2.13-4.4.0> Remove-Item "$path\taxi-trips-full-0" -Recurse -Force
PS G:\kafka_2.13-4.4.0> Remove-Item "$path\taxi-trips-full-1" -Recurse -Force
PS G:\kafka_2.13-4.4.0> Remove-Item "$path\taxi-trips-full-2" -Recurse -Force
PS G:\kafka_2.13-4.4.0> Get-ChildItem $path |
>> Where-Object { $_.Name -like "taxi-trips*" } |
>> Select-Object Name, FullName
PS G:\kafka_2.13-4.4.0>

In [2]:
kafka_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
taxi_df = spark.read.parquet(
    r"G:\NYC_Taxi_Project\yellow_tripdata_2026-08.parquet"
)

In [7]:
print("Total records:", taxi_df.count())
print("Columns:", taxi_df.columns)

taxi_df.printSchema()

Total records: 3336716
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'request_source']
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- 

In [5]:
from pyspark.sql.functions import col, struct, to_json

# Convert all taxi records into JSON Kafka messages
full_messages = taxi_df.select(
    to_json(
        struct(*[col(c) for c in taxi_df.columns]),
        {"ignoreNullFields": "false"}
    ).cast("binary").alias("value")
)

# Publish all records to Kafka
(
    full_messages.write
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("topic", "taxi-trips-full")
    .save()
)

print("Full dataset published to Kafka!")

Full dataset published to Kafka!


In [6]:
from pyspark.sql.functions import col, from_json

# Read taxi events from Kafka
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips-full")
    .option("startingOffsets", "earliest")
    .load()
)

print("Kafka streaming source initialized")

Kafka streaming source initialized


In [7]:
# Parse Kafka JSON messages using the original schema
parsed_stream = (
    kafka_stream
    .select(
        from_json(
            col("value").cast("string"),
            taxi_df.schema
        ).alias("data")
    )
    .select("data.*")
)

parsed_stream.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- request_source: string (nullable = true)



In [8]:
kafka_check = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips-full")
    .option("startingOffsets", "earliest")
    .load()
)

print("Kafka messages:", kafka_check.count())

Kafka messages: 6673432


In [12]:
print("Spark session:", spark)
print("Spark version:", spark.version)

Spark session: <pyspark.sql.session.SparkSession object at 0x0000028E53869E80>


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

dq_query = (
    parsed_stream.writeStream
    .foreachBatch(apply_dq_rules)
    .option(
        "checkpointLocation",
        "G:/NYC_Taxi_Project/checkpoints/taxi_dq"
    )
    .trigger(processingTime="10 seconds")
    .start()
)
def apply_dq_rules(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    print(f"Processing batch: {batch_id}")

    # Call your existing DQ transformation here.
    # Replace this line with the code that generates dq_df.
    dq_df = apply_existing_dq_logic(batch_df)

    # Inspect DQ results
    dq_df.groupBy("dq_status").count().show()

    # Your Delta writes will be added here

In [8]:
from pyspark.sql.functions import col, struct, to_json

# Convert each taxi record into a JSON Kafka message
test_messages = taxi_df.limit(100).select(
    to_json(
        struct(*[col(c) for c in taxi_df.columns]),
        {"ignoreNullFields": "false"}
    ).cast("binary").alias("value")
)

# Publish messages to Kafka
(
    test_messages.write
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("topic", "taxi-trips")
    .save()
)

print("Successfully sent 100 records to Kafka!")

Successfully sent 100 records to Kafka!


In [10]:
kafka_test = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips")
    .option("startingOffsets", "earliest")
    .load()
)

print("Messages received:", kafka_test.count())

kafka_test.select(
    col("value").cast("string").alias("json_record"),
    "partition",
    "offset"
).show(5, truncate=False)

Messages received: 100
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+
|json_record                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |partition

In [1]:
import os
import sys

os.environ["JAVA_HOME"] = (
    r"C:\Program Files\zulu17.70.201-ca-jdk17.0.21.1-win_x64"
    r"\zulu17.70.201-ca-jdk17.0.21.1-win_x64"
)

os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Limit Spark JVM heap at startup
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 512m pyspark-shell"

print("Java:", os.environ["JAVA_HOME"])
print("Python:", sys.executable)

Java: C:\Program Files\zulu17.70.201-ca-jdk17.0.21.1-win_x64\zulu17.70.201-ca-jdk17.0.21.1-win_x64
Python: c:\Users\HP\AppData\Local\Programs\Python\Python314\python.exe


In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[1]")
    .appName("NYC-Taxi-Kafka-Delta")

    # Delta Lake
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

    # Memory and processing controls
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.default.parallelism", "1")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.maxResultSize", "256m")
)

builder = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    ]
)

spark = builder.getOrCreate()

print("Spark version:", spark.version)
print(
    "JVM maximum heap:",
    round(
        spark.sparkContext._jvm.java.lang.Runtime
        .getRuntime().maxMemory() / (1024 ** 2),
        0
    ),
    "MB"
)

Spark version: 4.2.0
JVM maximum heap: 512.0 MB


In [3]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips-full")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "2000")
    .load()
)

kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, DoubleType,
    StringType, TimestampNTZType
)

taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("request_source", StringType(), True)
])

In [9]:
import os

parquet_path = r"G:\NYC_Taxi_Project\yellow_tripdata_2026-08.parquet"

print("File exists:", os.path.isfile(parquet_path))

File exists: True


In [10]:
taxi_df = spark.read.parquet(parquet_path)

print("Parquet loaded successfully")
print("Columns:", len(taxi_df.columns))

taxi_df.printSchema()

Parquet loaded successfully
Columns: 21
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- request_source: string (nullabl

In [13]:
from pyspark.sql.functions import col, struct, to_json

taxi_messages = (
    taxi_df
    .repartition(4)
    .select(
        to_json(
            struct(*[col(c) for c in taxi_df.columns]),
            {"ignoreNullFields": "false"}
        ).alias("value")
    )
)

In [14]:
print("Input partitions:", taxi_df.rdd.getNumPartitions())

Input partitions: 1


In [15]:
(
    taxi_messages.write
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("topic", "taxi-trips-full")
    .save()
)

In [16]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips-full")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "2000")
    .option("failOnDataLoss", "true")
    .load()
)

In [17]:
from pyspark.sql.functions import col, from_json

parsed_stream = (
    kafka_raw
    .select(
        from_json(
            col("value").cast("string"),
            taxi_schema
        ).alias("taxi")
    )
    .select("taxi.*")
)

In [ ]:
test_query = (
    parsed_stream.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option("numRows", 5)
    .option(
        "checkpointLocation",
        "G:/NYC_Taxi_Project/checkpoints/kafka_test_v2"
    )
    .trigger(availableNow=True)
    .start()
)



test_query.awaitTermination()

In [19]:
print("Query active:", test_query.isActive)
print("Query exception:", test_query.exception())
print("Last progress:", test_query.lastProgress)

Query active: False
Query exception: None
Last progress: {
    "id": "b00e6319-6d07-4b88-8544-5c590836d716",
    "runId": "829943a5-5334-4658-91dc-6a071d155088",
    "name": null,
    "timestamp": "2026-11-20T17:46:16.832Z",
    "batchId": 1668,
    "batchDuration": 1873,
    "numInputRows": 716,
    "inputRowsPerSecond": 533.1,
    "processedRowsPerSecond": 382.3,
    "durationMs": {
        "addBatch": 615,
        "commitOffsets": 613,
        "getBatch": 0,
        "latestOffset": 0,
        "queryPlanning": 5,
        "triggerExecution": 1873,
        "walCommit": 640
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[taxi-trips-full]]",
            "startOffset": {
                "taxi-trips-full": {
                    "0": 3336000
                }
            },
            "endOffset": {
                "taxi-trips-full": {
                    "0": 3336716
                }
            },
            "latestOffset": {
  

In [20]:
import os

bronze_path = "G:/NYC_Taxi_Project/delta/bronze/taxi_trips"

bronze_checkpoint = (
    "G:/NYC_Taxi_Project/checkpoints/kafka_bronze_v1"
)

print("Bronze path exists:", os.path.exists(bronze_path))
print("Checkpoint exists:", os.path.exists(bronze_checkpoint))

print(
    "Delta extension:",
    spark.conf.get("spark.sql.extensions")
)

print("Spark version:", spark.version)

Bronze path exists: False
Checkpoint exists: False
Delta extension: io.delta.sql.DeltaSparkSessionExtension
Spark version: 4.2.0


In [21]:

from pyspark.sql.functions import (
    col, when, lit, timestamp_diff, count
)
from pyspark.sql.window import Window

# Delta output paths
silver_path = "G:/NYC_Taxi_Project/delta/silver/taxi_trips"
quarantine_path = "G:/NYC_Taxi_Project/delta/quarantine/taxi_trips"

# New checkpoint for the DQ pipeline
dq_checkpoint = "G:/NYC_Taxi_Project/checkpoints/kafka_dq_v1"


def apply_dq_rules(batch_df, batch_id):

    # Keep original columns for duplicate detection
    original_columns = batch_df.columns

    # -----------------------------------------------------
    # Prepare derived field
    # -----------------------------------------------------

    dq_df = batch_df.withColumn(
        "trip_duration_minutes",
        timestamp_diff(
            "MINUTE",
            col("tpep_pickup_datetime"),
            col("tpep_dropoff_datetime")
        )
    )

    # DQ01 - Critical field completeness
    dq_df = dq_df.withColumn(
        "DQ01_critical_fields",
        when(
            col("tpep_pickup_datetime").isNull() |
            col("tpep_dropoff_datetime").isNull() |
            col("PULocationID").isNull() |
            col("DOLocationID").isNull(),
            "FAIL"
        ).otherwise("PASS")
    )

    # DQ02 - Timestamp consistency
    dq_df = dq_df.withColumn(
        "DQ02_timestamp_consistency",
        when(
            col("tpep_dropoff_datetime") <
            col("tpep_pickup_datetime"),
            "FAIL"
        ).otherwise("PASS")
    )

    # DQ03 - Duration plausibility
    dq_df = dq_df.withColumn(
        "DQ03_duration",
        when(col("trip_duration_minutes") < 0, "FAIL")
        .when(col("trip_duration_minutes") > 360, "WARNING")
        .otherwise("PASS")
    )

    # DQ04 - Distance plausibility
    dq_df = dq_df.withColumn(
        "DQ04_distance",
        when(col("trip_distance") < 0, "FAIL")
        .when(col("trip_distance") > 1000, "FAIL")
        .when(col("trip_distance") > 100, "WARNING")
        .otherwise("PASS")
    )

    # DQ05 - Categorical code validity
    dq_df = dq_df.withColumn(
        "DQ05_code_validity",
        when(
            col("VendorID").isNull() |
            ~col("VendorID").isin(1, 2, 6, 7),
            "FAIL"
        )
        .when(
            col("RatecodeID").isNotNull() &
            ~col("RatecodeID").isin(1, 2, 3, 4, 5, 6, 99),
            "FAIL"
        )
        .when(
            col("payment_type").isNull() |
            ~col("payment_type").isin(0, 1, 2, 3, 4, 5, 6),
            "FAIL"
        )
        .when(
            col("store_and_fwd_flag").isNotNull() &
            ~col("store_and_fwd_flag").isin("Y", "N"),
            "FAIL"
        )
        .otherwise("PASS")
    )

    # DQ06 - Source-aware completeness
    dq_df = dq_df.withColumn(
        "DQ06_source_completeness",
        when(
            col("request_source").isNull() &
            col("passenger_count").isNull(),
            "FAIL"
        ).otherwise("PASS")
    )

    # DQ07 - Exact duplicate detection within this microbatch
    duplicate_window = Window.partitionBy(
        *[col(c) for c in original_columns]
    )

    dq_df = dq_df.withColumn(
        "duplicate_count",
        count(lit(1)).over(duplicate_window)
    )

    dq_df = dq_df.withColumn(
        "DQ07_duplicate",
        when(col("duplicate_count") > 1, "FAIL")
        .otherwise("PASS")
    )

    # DQ08 - Location ID validity
    dq_df = dq_df.withColumn(
        "DQ08_location_validity",
        when(
            (col("PULocationID") < 1) |
            (col("PULocationID") > 265) |
            (col("DOLocationID") < 1) |
            (col("DOLocationID") > 265),
            "FAIL"
        ).otherwise("PASS")
    )

    # DQ09 - Passenger count validity
    dq_df = dq_df.withColumn(
        "DQ09_passenger_count",
        when(
            col("request_source").isNull() &
            col("PULocationID").isNotNull() &
            col("DOLocationID").isNotNull() &
            (col("passenger_count") == 0),
            "FAIL"
        ).otherwise("PASS")
    )

    # DQ10 - Negative payment detection
    money_columns = [
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "Airport_fee",
        "cbd_congestion_fee"
    ]

    negative_condition = None

    for c in money_columns:
        condition = col(c) < 0

        if negative_condition is None:
            negative_condition = condition
        else:
            negative_condition = negative_condition | condition

    dq_df = dq_df.withColumn(
        "DQ10_negative_payment",
        when(negative_condition, "WARNING")
        .otherwise("PASS")
    )

    # -----------------------------------------------------
    # Overall DQ status
    # -----------------------------------------------------

    fail_conditions = [
        col(c) == "FAIL"
        for c in [
            "DQ01_critical_fields",
            "DQ02_timestamp_consistency",
            "DQ03_duration",
            "DQ04_distance",
            "DQ05_code_validity",
            "DQ06_source_completeness",
            "DQ07_duplicate",
            "DQ08_location_validity",
            "DQ09_passenger_count"
        ]
    ]

    warning_conditions = [
        col(c) == "WARNING"
        for c in [
            "DQ03_duration",
            "DQ04_distance",
            "DQ10_negative_payment"
        ]
    ]

    any_fail = fail_conditions[0]
    for condition in fail_conditions[1:]:
        any_fail = any_fail | condition

    any_warning = warning_conditions[0]
    for condition in warning_conditions[1:]:
        any_warning = any_warning | condition

    dq_df = dq_df.withColumn(
        "dq_status",
        when(any_fail, "FAIL")
        .when(any_warning, "WARNING")
        .otherwise("PASS")
    )

    # Add microbatch identifier for traceability
    dq_df = dq_df.withColumn(
        "kafka_batch_id",
        lit(batch_id)
    )

    # -----------------------------------------------------
    # Route records to Delta Lake
    # -----------------------------------------------------

    silver_df = dq_df.filter(
        col("dq_status").isin("PASS", "WARNING")
    )

    quarantine_df = dq_df.filter(
        col("dq_status") == "FAIL"
    )

    # Append to Silver
    (
        silver_df.write
        .format("delta")
        .mode("append")
        .save(silver_path)
    )

    # Append to Quarantine
    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .save(quarantine_path)
    )

    print(f"Processed microbatch: {batch_id}")


In [22]:

dq_query = (
    parsed_stream.writeStream
    .foreachBatch(apply_dq_rules)
    .option("checkpointLocation", dq_checkpoint)
    .trigger(availableNow=True)
    .start()
)

print("DQ pipeline started:", dq_query.isActive)


DQ pipeline started: True


In [34]:
print("Active:", dq_query.isActive)
print("Status:", dq_query.status)
print("Progress:", dq_query.lastProgress)

Active: False
Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Progress: {
    "id": "b15ace00-6859-4021-b2ed-eb21673164b9",
    "runId": "3de21783-67ae-44f7-b8f0-60b43672d207",
    "name": null,
    "timestamp": "2026-11-21T08:09:15.864Z",
    "batchId": 1668,
    "batchDuration": 12166,
    "numInputRows": 1432,
    "inputRowsPerSecond": 134.8,
    "processedRowsPerSecond": 117.7,
    "durationMs": {
        "addBatch": 10331,
        "commitOffsets": 1114,
        "getBatch": 0,
        "latestOffset": 0,
        "queryPlanning": 7,
        "triggerExecution": 12166,
        "walCommit": 713
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[taxi-trips-full]]",
            "startOffset": {
                "taxi-trips-full": {
                    "0": 3336000
                }
            },
            "endOffset": {
                "taxi-trips-full": {
                    "0": 3336716
         

In [35]:
# Initialize a progress history dictionary
if "progress_history" not in globals():
    progress_history = {}

# Capture the currently available progress records
for p in dq_query.recentProgress:
    progress_history[p["batchId"]] = {
        "batch_id": p["batchId"],
        "records": p["numInputRows"],
        "input_rows_sec": p["inputRowsPerSecond"],
        "processed_rows_sec": p["processedRowsPerSecond"],
        "duration_ms": p["durationMs"].get("triggerExecution", 0)
    }

print("Captured batches:", len(progress_history))
print("Latest batch:", dq_query.lastProgress["batchId"])

Captured batches: 99
Latest batch: 1668


In [37]:
print("Active:", dq_query.isActive)
print("Exception:", dq_query.exception())
print("Last batch ID:", dq_query.lastProgress["batchId"])
print("Last batch records:", dq_query.lastProgress["numInputRows"])

Active: False
Exception: None
Last batch ID: 1668
Last batch records: 1432


In [39]:
progress = dq_query.lastProgress

print("Final batch ID:", progress["batchId"])
print("Final batch records:", progress["numInputRows"])
print("Final batch duration:", progress["durationMs"])
print("Input throughput:", progress["inputRowsPerSecond"])
print("Processing throughput:", progress["processedRowsPerSecond"])
print("Source offsets:", progress["sources"])

Final batch ID: 1668
Final batch records: 1432
Final batch duration: {'addBatch': 10331, 'commitOffsets': 1114, 'getBatch': 0, 'latestOffset': 0, 'queryPlanning': 7, 'triggerExecution': 12166, 'walCommit': 713}
Input throughput: 134.8
Processing throughput: 117.7
Source offsets: [{
    "description": "KafkaV2[Subscribe[taxi-trips-full]]",
    "startOffset": {
        "taxi-trips-full": {
            "0": 3336000
        }
    },
    "endOffset": {
        "taxi-trips-full": {
            "0": 3336716
        }
    },
    "latestOffset": {
        "taxi-trips-full": {
            "0": 3336716
        }
    },
    "numInputRows": 1432,
    "inputRowsPerSecond": 134.8,
    "processedRowsPerSecond": 117.7,
    "metrics": {
        "avgOffsetsBehindLatest": "0.0",
        "maxOffsetsBehindLatest": "0",
        "minOffsetsBehindLatest": "0"
    }
}]


In [41]:
silver_df = spark.read.format("delta").load(silver_path)
quarantine_df = spark.read.format("delta").load(quarantine_path)

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()

print("Silver records:", silver_count)
print("Quarantine records:", quarantine_count)
print("Total output records:", silver_count + quarantine_count)

print("Silver percentage:",
      round(silver_count / (silver_count + quarantine_count) * 100, 2))

print("Quarantine percentage:",
      round(quarantine_count / (silver_count + quarantine_count) * 100, 2))

Silver records: 3328863
Quarantine records: 7853
Total output records: 3336716
Silver percentage: 99.76
Quarantine percentage: 0.24


In [43]:

import subprocess
import re
from pathlib import Path
from datetime import datetime

# =====================================================
# CONFIGURATION — EXISTING PROJECT LOCATIONS
# =====================================================

KAFKA_HOME = Path(r"G:\kafka_2.13-4.4.0")
BOOTSTRAP = "localhost:9092"
TOPIC = "taxi-trips-full"

# =====================================================
# 1. READ KAFKA TOPIC METADATA
# =====================================================

def kafka_command(script, *args):
    command = [
        "cmd", "/c",
        str(KAFKA_HOME / "bin" / "windows" / script),
        *map(str, args)
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=60
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout)

    return result.stdout

topic_info = kafka_command(
    "kafka-topics.bat",
    "--bootstrap-server", BOOTSTRAP,
    "--describe",
    "--topic", TOPIC
)

partition_match = re.search(
    r"PartitionCount:\s*(\d+)", topic_info
)

partition_count = (
    int(partition_match.group(1))
    if partition_match else None
)

def read_offsets(time_value):
    output = kafka_command(
        "kafka-get-offsets.bat",
        "--bootstrap-server", BOOTSTRAP,
        "--topic", TOPIC,
        "--time", time_value
    )

    offsets = {}

    for line in output.strip().splitlines():
        parts = line.strip().split(":")
        if len(parts) >= 3:
            offsets[int(parts[-2])] = int(parts[-1])

    return offsets

earliest_offsets = read_offsets("-2")
latest_offsets = read_offsets("-1")

kafka_records = sum(
    latest_offsets.get(partition, 0) -
    earliest_offsets.get(partition, 0)
    for partition in latest_offsets
)

# =====================================================
# 2. READ DELTA LAKE RECORD COUNTS
# =====================================================

silver_df = spark.read.format("delta").load(silver_path)
quarantine_df = spark.read.format("delta").load(quarantine_path)

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()

total_output = silver_count + quarantine_count

# =====================================================
# 3. DELTA FILE STATISTICS
# =====================================================

def get_delta_stats(path):
    files = list(Path(path).rglob("*.parquet"))
    total_bytes = sum(f.stat().st_size for f in files)

    return {
        "file_count": len(files),
        "size_mb": total_bytes / (1024 ** 2),
        "avg_mb": (
            total_bytes / len(files) / (1024 ** 2)
            if files else 0
        )
    }

silver_stats = get_delta_stats(silver_path)
quarantine_stats = get_delta_stats(quarantine_path)

# =====================================================
# 4. SPARK EXECUTION METRICS
# =====================================================

progress = dq_query.lastProgress

if progress is None:
    raise RuntimeError("No Spark progress information available.")

duration = progress["durationMs"]
source = progress["sources"][0]

master = spark.sparkContext.master
final_batch = progress["batchId"]

# =====================================================
# 5. DISPLAY PERFORMANCE REPORT
# =====================================================

def show(label, value):
    print(f"{label:<37}: {value}")

print("\n" + "=" * 72)
print("       NYC TAXI — STREAMING DATA QUALITY PIPELINE")
print("                    PERFORMANCE REPORT")
print("=" * 72)

print("\n[ EXECUTION CONFIGURATION ]")
show("Execution status",
     "SUCCESS" if not dq_query.isActive
     and dq_query.exception() is None else "CHECK")
show("Spark master", master)
show("Kafka topic", TOPIC)
show("Kafka partitions", partition_count)
show("Kafka broker", BOOTSTRAP)

print("\n[ DATA INGESTION ]")
show("Kafka records available", f"{kafka_records:,}")
show("Kafka partition offsets", latest_offsets)
show("Spark final batch ID", final_batch)
show("Spark final batch records",
     f'{progress["numInputRows"]:,}')

print("\n[ PROCESSING PERFORMANCE ]")
show("Final batch execution time",
     f'{duration.get("triggerExecution", 0) / 1000:.2f} sec')
show("Final batch processing time",
     f'{duration.get("addBatch", 0) / 1000:.2f} sec')
show("Final batch input throughput",
     f'{progress["inputRowsPerSecond"]:.2f} records/sec')
show("Final batch processing throughput",
     f'{progress["processedRowsPerSecond"]:.2f} records/sec')
show("Final batch offset lag",
     source["metrics"].get("maxOffsetsBehindLatest", "N/A"))

print("\n[ DATA QUALITY RESULTS ]")
show("Total Delta output records", f"{total_output:,}")
show("Silver records", f"{silver_count:,}")
show("Quarantine records", f"{quarantine_count:,}")

if total_output:
    show("Silver acceptance rate",
         f"{silver_count / total_output * 100:.2f}%")
    show("Quarantine rate",
         f"{quarantine_count / total_output * 100:.2f}%")

print("\n[ DELTA LAKE STORAGE ]")
show("Silver Parquet files", silver_stats["file_count"])
show("Quarantine Parquet files", quarantine_stats["file_count"])
show("Silver storage", f'{silver_stats["size_mb"]:.2f} MB')
show("Quarantine storage", f'{quarantine_stats["size_mb"]:.2f} MB')
show("Total storage",
     f'{silver_stats["size_mb"] + quarantine_stats["size_mb"]:.2f} MB')
show("Average Silver file size",
     f'{silver_stats["avg_mb"]:.2f} MB')

print("\n[ RECONCILIATION ]")
show("Kafka vs Delta record count",
     "PASS" if kafka_records == total_output else "CHECK")
show("Silver + Quarantine",
     "PASS" if total_output == silver_count + quarantine_count
     else "CHECK")

print("\nGenerated:", datetime.now().strftime("%d-%b-%Y %H:%M:%S"))
print("=" * 72)



       NYC TAXI — STREAMING DATA QUALITY PIPELINE
                    PERFORMANCE REPORT

[ EXECUTION CONFIGURATION ]
Execution status                     : SUCCESS
Spark master                         : local[1]
Kafka topic                          : taxi-trips-full
Kafka partitions                     : 1
Kafka broker                         : localhost:9092

[ DATA INGESTION ]
Kafka records available              : 3,336,716
Kafka partition offsets              : {0: 3336716}
Spark final batch ID                 : 1668
Spark final batch records            : 1,432

[ PROCESSING PERFORMANCE ]
Final batch execution time           : 12.17 sec
Final batch processing time          : 10.33 sec
Final batch input throughput         : 134.80 records/sec
Final batch processing throughput    : 117.70 records/sec
Final batch offset lag               : 0

[ DATA QUALITY RESULTS ]
Total Delta output records           : 3,336,716
Silver records                       : 3,328,863
Quarantine records 

In [46]:

from pathlib import Path
import time

# Original Delta tables
silver_path = "G:/NYC_Taxi_Project/delta/silver/taxi_trips"
quarantine_path = "G:/NYC_Taxi_Project/delta/quarantine/taxi_trips"

# New experimental output paths
compact_silver_path = (
    "G:/NYC_Taxi_Project/delta/experiments/compacted/silver"
)
compact_quarantine_path = (
    "G:/NYC_Taxi_Project/delta/experiments/compacted/quarantine"
)

# Safety check: never overwrite an existing experiment
for path in [compact_silver_path, compact_quarantine_path]:
    if Path(path).exists():
        raise FileExistsError(
            f"Experiment output already exists: {path}\n"
            "Use a new experiment folder before running again."
        )

def get_delta_stats(path):
    files = list(Path(path).rglob("*.parquet"))
    total_bytes = sum(f.stat().st_size for f in files)

    return {
        "files": len(files),
        "size_mb": total_bytes / (1024 * 1024),
        "avg_file_mb": (
            total_bytes / len(files) / (1024 * 1024)
            if files else 0
        )
    }

# Capture original file statistics
silver_before = get_delta_stats(silver_path)
quarantine_before = get_delta_stats(quarantine_path)

print("Starting Delta compaction experiment...")

start_time = time.time()

# Compact Silver: consolidate into 4 output partitions
silver_df = spark.read.format("delta").load(silver_path)

(
    silver_df
    .coalesce(4)
    .write
    .format("delta")
    .mode("errorifexists")
    .save(compact_silver_path)
)

# Compact Quarantine: consolidate into 2 output partitions
quarantine_df = spark.read.format("delta").load(quarantine_path)

(
    quarantine_df
    .coalesce(2)
    .write
    .format("delta")
    .mode("errorifexists")
    .save(compact_quarantine_path)
)

compaction_time = time.time() - start_time

# Verify record counts
silver_original_count = silver_df.count()
quarantine_original_count = quarantine_df.count()

silver_compact_count = (
    spark.read.format("delta")
    .load(compact_silver_path)
    .count()
)

quarantine_compact_count = (
    spark.read.format("delta")
    .load(compact_quarantine_path)
    .count()
)

# Capture compacted file statistics
silver_after = get_delta_stats(compact_silver_path)
quarantine_after = get_delta_stats(compact_quarantine_path)

print("\n========== DELTA COMPACTION RESULTS ==========")

print(f"Compaction time: {compaction_time:.2f} seconds")

print("\nSILVER")
print(f"Original files:       {silver_before['files']}")
print(f"Compacted files:      {silver_after['files']}")
print(f"Original size:        {silver_before['size_mb']:.2f} MB")
print(f"Compacted size:       {silver_after['size_mb']:.2f} MB")
print(f"Original avg file:    {silver_before['avg_file_mb']:.2f} MB")
print(f"Compacted avg file:   {silver_after['avg_file_mb']:.2f} MB")
print(f"Original records:     {silver_original_count:,}")
print(f"Compacted records:    {silver_compact_count:,}")

print("\nQUARANTINE")
print(f"Original files:       {quarantine_before['files']}")
print(f"Compacted files:      {quarantine_after['files']}")
print(f"Original size:        {quarantine_before['size_mb']:.2f} MB")
print(f"Compacted size:       {quarantine_after['size_mb']:.2f} MB")
print(f"Original records:     {quarantine_original_count:,}")
print(f"Compacted records:    {quarantine_compact_count:,}")

before_files = silver_before["files"] + quarantine_before["files"]
after_files = silver_after["files"] + quarantine_after["files"]

reduction = (
    (before_files - after_files) / before_files * 100
    if before_files else 0
)

print("\nOVERALL")
print(f"Files before:         {before_files:,}")
print(f"Files after:          {after_files:,}")
print(f"File reduction:       {reduction:.2f}%")

print(
    "\nRecord validation:",
    "PASS" if (
        silver_original_count == silver_compact_count
        and quarantine_original_count == quarantine_compact_count
    ) else "FAIL"
)

print("==============================================")


Starting Delta compaction experiment...

========== DELTA COMPACTION RESULTS ==========
Compaction time: 296.20 seconds

SILVER
Original files:       1835
Compacted files:      4
Original size:        179.70 MB
Compacted size:       85.15 MB
Original avg file:    0.10 MB
Compacted avg file:   21.29 MB
Original records:     3,328,863
Compacted records:    3,328,863

QUARANTINE
Original files:       1834
Compacted files:      2
Original size:        73.54 MB
Compacted size:       0.24 MB
Original records:     7,853
Compacted records:    7,853

OVERALL
Files before:         3,669
Files after:          6
File reduction:       99.84%

Record validation: PASS


In [47]:

from pathlib import Path

# New experiment paths
experiment_root = "G:/NYC_Taxi_Project/delta/experiments/batch_5000"

experiment_silver_path = f"{experiment_root}/silver"
experiment_quarantine_path = f"{experiment_root}/quarantine"

experiment_checkpoint = (
    "G:/NYC_Taxi_Project/checkpoints/kafka_dq_5000"
)

# Safety checks: prevent accidental overwrite or duplicate runs
paths_to_check = [
    experiment_silver_path,
    experiment_quarantine_path,
    experiment_checkpoint
]

existing_paths = [p for p in paths_to_check if Path(p).exists()]

if existing_paths:
    raise FileExistsError(
        "Experiment folders already exist:\n" +
        "\n".join(existing_paths) +
        "\nUse a fresh experiment folder/checkpoint before rerunning."
    )

# Redirect the existing DQ function to experiment outputs
silver_path = experiment_silver_path
quarantine_path = experiment_quarantine_path
dq_checkpoint = experiment_checkpoint

print("Batch-size experiment configured.")
print("Batch size: 5000")
print("Silver:", silver_path)
print("Quarantine:", quarantine_path)
print("Checkpoint:", dq_checkpoint)


Batch-size experiment configured.
Batch size: 5000
Silver: G:/NYC_Taxi_Project/delta/experiments/batch_5000/silver
Quarantine: G:/NYC_Taxi_Project/delta/experiments/batch_5000/quarantine
Checkpoint: G:/NYC_Taxi_Project/checkpoints/kafka_dq_5000


In [48]:

# Start a fresh Kafka stream with the existing parsed schema/dataframe.
# A new checkpoint allows replay from the beginning.

experiment_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxi-trips-full")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "5000")
    .load()
)

from pyspark.sql.functions import from_json

experiment_parsed_stream = (
    experiment_stream
    .select(
        from_json(col("value").cast("string"), taxi_schema).alias("data")
    )
    .select("data.*")
)

import time

experiment_start_time = time.time()

batch_5000_query = (
    experiment_parsed_stream.writeStream
    .foreachBatch(apply_dq_rules)
    .option("checkpointLocation", experiment_checkpoint)
    .trigger(availableNow=True)
    .start()
)

print("5,000 batch-size experiment started.")


5,000 batch-size experiment started.


In [49]:

batch_5000_query.awaitTermination()

experiment_end_time = time.time()
experiment_duration = experiment_end_time - experiment_start_time

print("Experiment completed:", not batch_5000_query.isActive)
print("Exception:", batch_5000_query.exception())
print("Total execution time:", round(experiment_duration, 2), "seconds")
print("Final batch ID:", batch_5000_query.lastProgress["batchId"])
print("Last batch metrics:")
print(batch_5000_query.lastProgress)

\




Processed microbatch: 0
Processed microbatch: 1
Processed microbatch: 2
Processed microbatch: 3
Processed microbatch: 4
Processed microbatch: 5
Processed microbatch: 6
Processed microbatch: 7
Processed microbatch: 8
Processed microbatch: 9
Processed microbatch: 10
Processed microbatch: 11
Processed microbatch: 12
Processed microbatch: 13
Processed microbatch: 14
Processed microbatch: 15
Processed microbatch: 16
Processed microbatch: 17
Processed microbatch: 18
Processed microbatch: 19
Processed microbatch: 20
Processed microbatch: 21
Processed microbatch: 22
Processed microbatch: 23
Processed microbatch: 24
Processed microbatch: 25
Processed microbatch: 26
Processed microbatch: 27
Processed microbatch: 28
Processed microbatch: 29
Processed microbatch: 30
Processed microbatch: 31
Processed microbatch: 32
Processed microbatch: 33
Processed microbatch: 34
Processed microbatch: 35
Processed microbatch: 36
Processed microbatch: 37
Processed microbatch: 38
Processed microbatch: 39
Processed 

In [50]:

from pathlib import Path

# Read experiment outputs
silver_5000 = (
    spark.read.format("delta")
    .load(experiment_silver_path)
)

quarantine_5000 = (
    spark.read.format("delta")
    .load(experiment_quarantine_path)
)

# Count records
silver_count_5000 = silver_5000.count()
quarantine_count_5000 = quarantine_5000.count()

total_count_5000 = silver_count_5000 + quarantine_count_5000

expected_records = 3336716

# Calculate metrics
total_seconds = experiment_duration
average_throughput = total_count_5000 / total_seconds

print("=" * 55)
print("       BATCH SIZE 5000 — VALIDATION REPORT")
print("=" * 55)

print(f"Silver records       : {silver_count_5000:,}")
print(f"Quarantine records   : {quarantine_count_5000:,}")
print(f"Total Delta records  : {total_count_5000:,}")
print(f"Expected Kafka records: {expected_records:,}")

print(f"\nExecution time       : {total_seconds:.2f} seconds")
print(f"Average throughput   : {average_throughput:.2f} records/sec")

print("\nRECONCILIATION")
print(
    "Kafka vs Delta       :",
    "PASS" if total_count_5000 == expected_records else "FAIL"
)

print(
    "Silver + Quarantine :",
    "PASS" if (
        silver_count_5000 + quarantine_count_5000
        == total_count_5000
    ) else "FAIL"
)

print("=" * 55)


       BATCH SIZE 5000 — VALIDATION REPORT
Silver records       : 3,328,863
Quarantine records   : 7,853
Total Delta records  : 3,336,716
Expected Kafka records: 3,336,716

Execution time       : 7780.88 seconds
Average throughput   : 428.84 records/sec

RECONCILIATION
Kafka vs Delta       : PASS
Silver + Quarantine : PASS


In [7]:
print("Query active:", kafka_test_query.isActive)
print("Query status:", kafka_test_query.status)

print("\n--- EXCEPTION DETAILS ---")
print(kafka_test_query.exception())

Query active: False
Query status: {'message': "Terminated with exception: Some data may have been lost because they are not available in Kafka any more;\neither the data was aged out by Kafka or the topic may have been deleted before all the data in the\ntopic was processed.\nIf you don't want your streaming query to fail on such cases, set the source option failOnDataLoss to false.\nReason: Could not read records in offset [0, 1112208) for topic partition taxi-trips-full-0\nwith consumer group spark-kafka-source-238d9db1-a66d-4dd6-9b37-8adc4edfdced--1572205181-executor.", 'isDataAvailable': False, 'isTriggerActive': False}

--- EXCEPTION DETAILS ---
Some data may have been lost because they are not available in Kafka any more;
either the data was aged out by Kafka or the topic may have been deleted before all the data in the
topic was processed.
If you don't want your streaming query to fail on such cases, set the source option failOnDataLoss to false.
Reason: Could not read records i